In [1]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

class PreprocessingModule:
    def __init__(self, remove_stopwords=True, lemmatize=True):
        self.remove_stopwords = remove_stopwords
        self.lemmatize = lemmatize
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def transform(self, text):
        # Edge Case Handling
        if not text or text.strip() == "":
            raise ValueError("Empty input text provided")

        if len(text.strip()) == 1:
            raise ValueError("Query too short (single character)")

        if re.fullmatch(r'[\W\d_]+', text):
            raise ValueError("Input contains only numbers/symbols")

        # Lowercase
        text = text.lower()

        # Remove punctuation & numbers
        text = re.sub(r'[^a-z\s]', '', text)

        tokens = text.split()

        if self.remove_stopwords:
            tokens = [word for word in tokens if word not in self.stop_words]

        if self.lemmatize:
            tokens = [self.lemmatizer.lemmatize(word) for word in tokens]

        return " ".join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class VectorizerModule:
    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.corpus_vectors = None

    def fit(self, corpus):
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)

    def transform(self, query):
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.corpus_vectors)
        return similarities.flatten()

In [3]:
class Pipeline:
    def __init__(self):
        self.preprocessor = PreprocessingModule()
        self.vectorizer = VectorizerModule()

    def run(self, query, corpus):
        try:
            # Preprocess corpus
            processed_corpus = [self.preprocessor.transform(doc) for doc in corpus]

            # Fit vectorizer
            self.vectorizer.fit(processed_corpus)

            # Preprocess query
            processed_query = self.preprocessor.transform(query)

            # Get similarity scores
            scores = self.vectorizer.transform(processed_query)

            # Rank results
            ranked_results = sorted(
                list(zip(corpus, scores)),
                key=lambda x: x[1],
                reverse=True
            )

            return ranked_results

        except ValueError as e:
            return f"Error: {str(e)}"

In [4]:
corpus = [
    "Machine learning is amazing",
    "Deep learning uses neural networks",
    "Natural language processing deals with text",
    "AI is transforming industries",
    "Python is great for data science",
    "Football is a popular sport",
    "Cricket is widely played in India",
    "Basketball requires teamwork",
    "Cooking requires ingredients and skills",
    "Baking cakes is fun",
    "Traveling opens new experiences",
    "Mountains are beautiful",
    "Beaches are relaxing",
    "Music soothes the soul",
    "Art expresses creativity"
]

In [5]:
pipeline = Pipeline()

queries = [
    "AI and machine learning",
    "sports like cricket",
    "cooking food",
    "beautiful places to travel",
    "music and art"
]

for q in queries:
    print(f"\nQuery: {q}")
    results = pipeline.run(q, corpus)

    if isinstance(results, str):
        print(results)
    else:
        for doc, score in results[:5]:
            print(f"{score:.4f} -> {doc}")


Query: AI and machine learning
0.6369 -> Machine learning is amazing
0.3479 -> AI is transforming industries
0.2084 -> Deep learning uses neural networks
0.0000 -> Natural language processing deals with text
0.0000 -> Python is great for data science

Query: sports like cricket
0.4082 -> Football is a popular sport
0.3536 -> Cricket is widely played in India
0.0000 -> Machine learning is amazing
0.0000 -> Deep learning uses neural networks
0.0000 -> Natural language processing deals with text

Query: cooking food
0.5161 -> Cooking requires ingredients and skills
0.0000 -> Machine learning is amazing
0.0000 -> Deep learning uses neural networks
0.0000 -> Natural language processing deals with text
0.0000 -> AI is transforming industries

Query: beautiful places to travel
0.7071 -> Mountains are beautiful
0.0000 -> Machine learning is amazing
0.0000 -> Deep learning uses neural networks
0.0000 -> Natural language processing deals with text
0.0000 -> AI is transforming industries

Query:

In [6]:
edge_cases = ["", "a", "12345!!!"]

for case in edge_cases:
    print(f"\nTesting edge case: '{case}'")
    print(pipeline.run(case, corpus))


Testing edge case: ''
Error: Empty input text provided

Testing edge case: 'a'
Error: Query too short (single character)

Testing edge case: '12345!!!'
Error: Input contains only numbers/symbols
